# Week 4: does the signal show up?

**Question.** Can order-book imbalance and order-flow imbalance predict short-term changes in the mid price in a simulated market?

There are two questions hiding in that sentence. Does the price move *at the same moment* as the imbalance (price impact)? Does the imbalance tell you what the price will do *next*, on data your model has not seen (prediction)? This notebook is about telling them apart.

**Target.** $y_t = \mathrm{sign}(m_{t+10} - m_t)$, ten commands ahead. Classes: $-1$ down, $0$ unchanged, $+1$ up.

**Rules.** Features at row $t$ use rows up to $t$ only. Never use the simulator's hidden fair value. A null result is a fine result. Predicting well on synthetic data does not show a real trading edge.

Work top to bottom. Cells marked **TODO** are yours.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # run the notebook from the notebooks/ folder
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix

from research import features as F
from research.make_tapes import MODES, SEEDS, tape_path
from research.simulator import SimConfig, write_tape
from research.record import replay

HORIZON = 10

## 1. Get the tapes and check your engine

Tapes are generated from a seed, so everyone gets the same files. Then run `python -m research.check_replay` in a terminal. It must say PASSED before you trust any result below.

In [ ]:
for mode in MODES:
    for seed in SEEDS:
        if not os.path.exists(tape_path(mode, seed)):
            write_tape(SimConfig(seed=seed, mode=mode), tape_path(mode, seed))
print("tapes ready:", [os.path.basename(tape_path(m, s)) for m in MODES for s in SEEDS])

## 2. Replay through your engine

This records the book after every command, with the command index `idx`. It takes a few seconds per tape.

In [ ]:
states = {}
for mode in MODES:
    for seed in SEEDS:
        rows, digest = replay(open(tape_path(mode, seed), encoding="utf-8").read())
        df = pd.DataFrame(rows).set_index("idx")
        states[(mode, seed)] = df
        print(mode, seed, len(df), "commands, checksum", digest[:12])

## 3. Data quality

**TODO.** For each tape, report: the share of commands where the book has no valid mid (empty side or crossed), the share of rejected commands, trades per command, and the typical spread. Decide how you will treat rows without a mid and write the rule down.

In [ ]:
# TODO: build a small table, one row per (mode, seed).
quality = pd.DataFrame()
quality

## 4. Features

Implement the functions in `research/features.py` (check yourself with `python -m unittest research.test_features -v`), then build the feature table.

In [ ]:
data = {key: F.build(df, HORIZON) for key, df in states.items()}
data[("signal", 1)].describe().T

**TODO.** Plot at least two features against the label or the future price change, for one control and one signal tape. Say what you see, and what you cannot conclude from the plot.

In [ ]:
# TODO: feature plots

## 5. Labels and the chronological split

60% train, 20% validation, 20% test, in time order, inside each tape. Labels that look across a boundary are purged. Fit scalers on train only. Open the test part once, at the end.

In [ ]:
def sets(X):
    sp = F.chrono_split(len(X), 0.6, 0.2, HORIZON)
    D = X.dropna(subset=F.FEATURE_COLUMNS + ["y"])
    return {k: D[D.index.isin(v)] for k, v in sp.items()}

split = {key: sets(X) for key, X in data.items()}
# TODO: class counts per part, for one control and one signal tape.

## 6. Models

Three are required: the **majority class** baseline, an **imbalance threshold rule** (predict up when imbalance is above a threshold, down when below its negative, otherwise unchanged; choose the threshold on validation), and **logistic regression**. Random forest or walk-forward are optional extras.

In [ ]:
def report(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "balanced_acc": balanced_accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro")}

def fit_and_score(parts):
    train, val, test = parts["train"], parts["val"], parts["test"]
    # TODO: majority baseline, threshold rule (threshold chosen on val), logistic regression
    # (fit the scaler on train only). Return {model_name: report(...)} on the TEST part.
    return {}

## 7. Control versus signal, across seeds

**TODO.** Run `fit_and_score` on every tape and put the results side by side. Does the model beat the baselines? Is it the same in the control market? Is it stable across seeds?

In [ ]:
results = {key: fit_and_score(parts) for key, parts in split.items()}
# TODO: tidy table of results, plus a confusion matrix for logistic regression on one signal tape.

## 8. Interpretation and limits

Write a few honest paragraphs.

* What did you find in the control market, and what in the signal market?
* Which of your results is price impact at the same moment, and which is prediction on unseen data?
* How did you check for leakage?
* What could still be wrong? What would change your mind?
* How to reproduce everything: the seeds, settings and commands.